> 在线讲义：[第 55 章 · 扩缩容、故障恢复与部署方案比较：积压多久消退](https://wikieden.github.io/ai-infra-from-constraints/lessons/55-scale-compare.html) · 本书《从约束推导 AI Infra》

# 第 55 章 · 扩缩容、故障恢复与部署方案比较：积压多久消退

配套讲义：仓库里 `ai-infra/notes/lessons/55-scale-compare.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



启动积压的排空、后台迁移追赶、重配置、输出记录恢复与例 9.8 的方案比较，取自 AII 第 9 章 §9.6–§9.7。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
GiB = 2**30
K_TOKEN = 147456
MU = {"八个完整副本": 3.018, "理想分块共置": 4.17, "直接 PD": 4 / 0.8782}

**函数 `drain_s`**：T_drain = λ T_s / (μ − λ)；μ ≤ λ 时返回无穷。

In [ ]:
def drain_s(lam: float, Ts: float, mu: float) -> float:
    """T_drain = λ T_s / (μ − λ)；μ ≤ λ 时返回无穷。"""
    return lam * Ts / (mu - lam) if mu > lam else float("inf")

**函数 `min_rate`**：在 deadline 前清空启动积压所需的最低服务率。

In [ ]:
def min_rate(lam: float, Ts: float, deadline: float) -> float:
    """在 deadline 前清空启动积压所需的最低服务率。"""
    return lam + lam * Ts / (deadline - Ts)

**函数 `catch_up_s`**：后台复制追上仍在增长的状态：state / (copy − growth)。

In [ ]:
def catch_up_s(state: float, copy_bw: float, growth: float) -> float:
    """后台复制追上仍在增长的状态：state / (copy − growth)。"""
    return state / (copy_bw - growth)

**函数 `breakeven_steps`**：小工具。

In [ ]:
def breakeven_steps(Ts: float, delta: float) -> int:
    return round(Ts / delta)

**函数 `cost_per_k`**：每千个合格请求的成本。

In [ ]:
def cost_per_k(yuan_per_h: float, rate: float, qualified: float = 1.0) -> float:
    """每千个合格请求的成本。"""
    return yuan_per_h / (rate * qualified * 3600) * 1000

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：启动与预热")
print(f"  启动多 10 s、每步省 0.2 ms：{breakeven_steps(10, 0.2e-3):,} 步持平（batch 32 约 {breakeven_steps(10, 0.2e-3) * 32 / 1e6:.1f} M token）")
for name, mu in MU.items():
    d = drain_s(3.5, 10, mu)
    print(f"  {name} {mu:.2f} 请求/s：" + (f"就绪后 {d:.1f} s 排空，第 {10 + d:.1f} 秒" if d != float("inf") else "积压一直增长"))

In [ ]:
print("\n第二步：迁移与重配置")
growth = 1166 * K_TOKEN / 1e9
state = 32 * 8704 * K_TOKEN / 1e9
for name, bw in (("200 Gbit/s 25 GB/s", 25.0), ("50 GbE 6.25 GB/s", 6.25)):
    t = catch_up_s(state, bw, growth)
    print(f"  {state:.1f} GB、每秒长 {growth:.3f} GB，{name}：{t:.2f} s 追上，期间新增 {t * growth:.2f} GB")
for name, bw in (("50 GbE", 6.25e9), ("200 Gbit/s", 25e9)):
    print(f"  TP4 → TP8 16.4 GB 走 {name}：{16.4e9 / bw:.2f} s，加 9 s 准备 {16.4e9 / bw + 9:.1f} s")
print(f"  同机 NVLink 最忙卡 4.7 GB / 300 GB/s = {4.7e9 / 300e9 * 1e3:.1f} ms，总时间仍约 9 s")

In [ ]:
print("\n第三步：恢复")
print("  已返回 1025 个输出、KV 只存了输入：把前 1024 个输出当 prefill 补算，重建 9216 个 token 的 KV")

In [ ]:
print("\n第四步：例 9.8")
v = 1.125 * GiB
for name, mult in (("直接 PD", 1), ("PD + 共享池", 2)):
    print(f"  {name}：每请求 {mult * 1.125} GiB，通道 {25e9 / (mult * v):.1f} 请求/s；3.5 请求/s 时 {3.5 * mult * v / 1e9:.1f} GB/s")
need = min_rate(3.5, 10, 60)
print(f"  60 s 内清空：μ ≥ {need:.1f}；直接 PD 余量 {(MU['直接 PD'] / need - 1) * 100:.0f}%")
mu9 = MU["直接 PD"] * 0.9
print(f"  效率只有九成：{mu9:.2f} 请求/s → 第 {10 + drain_s(3.5, 10, mu9):.1f} 秒才清空")
print(f"  完整副本到第 60 秒积压 {35 + 50 * (3.5 - MU['八个完整副本']):.0f} 个")
print(f"  成本：每小时 8 元、3.5 请求/s → {cost_per_k(8, 3.5):.2f} 元/千请求；满载 3.02 / 4.55 → {cost_per_k(8, 3.02):.2f} / {cost_per_k(8, 4.55):.2f}；只有 1/4 合格 → {cost_per_k(8, 4.55, 0.25):.2f}")

In [ ]:
print("\n第五步：负载变了")
print(f"  后续复用一次：中转 + 取回 96.6 ms vs 重算 856 ms → 省 {856 - 96.6:.0f} ms；每对搬 3.375 GiB，3.5 对/s 要 {3.5 * 3.375 * GiB / 1e9:.1f} GB/s，上限 {25e9 / (3.375 * GiB):.1f} 对/s")
for name, mu in (("完整副本", 4.897), ("原 PD 4P/4D", MU["直接 PD"]), ("2P/6D", 5.69)):
    print(f"  命中 6144 前缀 · {name} {mu:.2f}：第 {10 + drain_s(3.5, 10, mu):.1f} 秒清空")
print("  输出 4097：最佳 2P/6D 只有 1.26 请求/s → 持续要 3 组（3.78），按期清空要 4 组（5.04）")

## 3. 代码里才看得到的

1. 理想分块共置用原文的 4.17 算出第 62.2 秒，原文写"第 63 秒"，是向上取整的说法；两种说法都错过第 60 秒。
2. 直接 PD 的 μ 用 4 / 0.8782 = 4.555，不是 4.55，所以九成效率排空是第 68.4 秒（原文 68.5）。
3. `min_rate` 里期限 60 s 包含启动的 10 s，真正用来消化积压的只有 50 s。

## 4. 自己改一改

- 给 `drain_s` 加突发：第 20 秒多来 20 个请求，看直接 PD 还赶不赶得上第 60 秒。
- 用 `catch_up_s` 找"永远追不上"的链路带宽。
- 把 `cost_per_k` 换成按 GPU 秒计费：每 GPU 秒 x 元，比较按小时预留与按用量。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 收费站关了再开，长龙按放行减来车的速度消；
> 房东还在住先搬不动的，最后停一下交钥匙；搬得再快也得等装修；
> 直播断线，观众看到哪就从哪讲；
> 先看搬得完，再看按期，最后按合格的单子算价钱。